## Preprocessing

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "workflows"))
from c21_surrogate_training import run_preprocessing

# ── Parameters ───────────────────────────────────────────────────────────────
EDGE_CSV        = "v6_edge_C14_S19999_D20260516"
NODE_CSV        = "v6_node_C12_S19999_D20260516"
BIDIRECTIONAL   = False    # True=240 edges (undirected), False=120 edges (best)
BATCH_SIZE      = 64       # was 32; larger batches → more stable gradients
DATA_INSPECTION = False    # print per-sample statistics

# Model architecture
# hidden_dim=64 / 4 layers → ~1.1M params (safe with weight_decay + dropout)
# num_layers=4: truss load paths span up to 7-8 hops — 3 layers couldn't see the full graph
HIDDEN_DIM      = 64       # keep; avoids overfitting risk
NUM_LAYERS      = 4        # was 3; better global force-path propagation
DROPOUT_P       = 0.3      # keep

# ── Run ───────────────────────────────────────────────────────────────────────
pre = run_preprocessing(
    EDGE_CSV, NODE_CSV,
    bidirectional   = BIDIRECTIONAL,
    batch_size      = BATCH_SIZE,
    data_inspection = DATA_INSPECTION,
    hidden_dim      = HIDDEN_DIM,
    num_layers      = NUM_LAYERS,
    dropout_p       = DROPOUT_P,
)

c:\Users\Jasper\Documents\PyEnvs\thesis_home_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config System loaded successfully, Code running locally from thesis_generative_timber and Data is connected to OneDrive 2.2 - 2.4.

Sample ID column: 'sample_id'
Topology: 120 edges in JSON | 120 rows/sample in CSV | 39 nodes/sample.
Using directed graph: 120 edges.
Final edge count: 120 (unidirectional).
Found 20000 matching samples.
Split: Train=16000 | Val=2000 | Test=2000
Norm stats saved: C:\Users\Jasper\Documents\PyRepo\thesis_generative_timber\02_data_io\norm_stats.pt
Train positive rate: 0.1958 → pos_weight=4.1080
Dataset: 16000 train | 2000 val | 2000 test. Each: 39 nodes, 120 edges.
Model on cuda | node_dim=10 | edge_dim=9 | hidden_dim=64 | num_layers=4 | dropout_p=0.3
DataLoaders: 250 train | 32 val | 32 test batches (batch_size=64)
WeightedBCELoss(pos_weight=4.1080)


## Training

In [ ]:
from c21_surrogate_training import run_training

# ── Parameters ───────────────────────────────────────────────────────────────
EPOCHS            = 200
LR                = 1e-4
PATIENCE          = 60
LR_FACTOR         = 0.5
LR_PATIENCE       = 15
LR_MIN            = 1e-6
GRAD_CLIP         = 1.0
WEIGHT_DECAY      = 1e-3
DEFAULT_THRESHOLD = 0.35
MIN_PRECISION     = 0.40
# Override auto-computed pos_weight (≈4.1 from class balance).
# Lower value → fewer false positives → better design-level discrimination.
# Trade-off: recall drops from ~88% to ~70%; precision rises from ~40% to ~55-60%.
POS_WEIGHT        = 2.5    # was auto ~4.1

# ── Run ───────────────────────────────────────────────────────────────────────
train_out = run_training(
    pre,
    epochs            = EPOCHS,
    lr                = LR,
    patience          = PATIENCE,
    lr_factor         = LR_FACTOR,
    lr_patience       = LR_PATIENCE,
    lr_min            = LR_MIN,
    grad_clip         = GRAD_CLIP,
    weight_decay      = WEIGHT_DECAY,
    default_threshold = DEFAULT_THRESHOLD,
    min_precision     = MIN_PRECISION,
    pos_weight        = POS_WEIGHT,
)

WeightedBCELoss(pos_weight=2.5000)

Hyperparameters: epochs=200, lr=1e-04, patience=60, grad_clip=1.0, weight_decay=1e-03, pos_weight=2.5000, default_threshold=0.35, min_precision=0.4
Checkpoint: C:\Users\Jasper\Documents\PyRepo\thesis_generative_timber\02_data_io\surrogate_v4_checkpoint.pth

Starting training: 200 epochs, early stopping patience=60
----------------------------------------------------------------------
Epoch 005  train=0.711416  val=0.693505  lr=1.00e-04  no_improve=0/60
Epoch 010  train=0.671485  val=0.657060  lr=1.00e-04  no_improve=0/60
Epoch 015  train=0.654360  val=0.642596  lr=1.00e-04  no_improve=0/60
Epoch 020  train=0.645080  val=0.633985  lr=1.00e-04  no_improve=0/60
Epoch 025  train=0.638057  val=0.627696  lr=1.00e-04  no_improve=0/60
Epoch 030  train=0.632507  val=0.620561  lr=1.00e-04  no_improve=0/60
Epoch 035  train=0.627791  val=0.617334  lr=1.00e-04  no_improve=0/60
Epoch 040  train=0.624344  val=0.614244  lr=1.00e-04  no_improve=1/60
Epoch 045  train=

## Evaluation

In [ ]:
from c21_surrogate_training import run_evaluation

# ── Run ───────────────────────────────────────────────────────────────────────
eval_out = run_evaluation(train_out, pre)


## Export

In [ ]:
from c21_surrogate_training import run_export

# ── Run ───────────────────────────────────────────────────────────────────────
export_out = run_export(pre, train_out, eval_out)
print("Saved to:", export_out["models_dir"])
print("Files:")
for f in export_out["all_files"]:
    print(" ", f)
